# Prompting a Local LLM

## Google Colab edition

**Applied AI Summer Workshop.**

A backup version of the workshop's local-LLM modules, for use if the
JupyterHub isn't available. It runs a language model on a Colab machine and
walks through four parts:

| Part | Topic |
| --- | --- |
| **A** | Setup — install Ollama, download a model |
| **B** | Prompting basics — the generation parameters |
| **C** | System prompts, model limits, and tool use |
| **D** | Structured output — getting clean JSON back |

The RAG material is a separate notebook.

## How this differs from the Hub version

The Hub version is a **marimo** notebook, with sliders and buttons that update
live. Colab runs Jupyter notebooks, which don't have those — so here you
**edit a variable at the top of a cell and re-run it** instead. Same ideas,
one extra keystroke.

## ⚠️ One important difference: this is not private

The original modules open with "nothing leaves this machine." On the Hub, or
on your own laptop, that's true.

**On Colab it is not.** This notebook runs on a Google virtual machine. Your
prompts travel to Google's servers, and you need a Google account to be here
at all. The model weights still run "locally" in the sense that no
OpenAI/Anthropic API is involved — but the privacy argument for local models
does not apply to this environment.

That's worth understanding rather than glossing over: *where the computation
happens* is a separate question from *whose model you are running*.

## Before you start: turn on the GPU

Colab gives you a free GPU, which makes the model noticeably faster.

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

It works without a GPU too, just more slowly.

# Part A — Setup

Run these cells **in order**, top to bottom. Together they take about 3–5
minutes, mostly downloading.

You'll need to re-run all of Part A every time Colab gives you a fresh
session — the machine is wiped between sessions, so nothing persists.

### A1. Install [Ollama](https://ollama.com)

Ollama is the program that downloads, stores, and runs models. Colab gives us
root access on a throwaway virtual machine (VM), so the standard install
script works.

One wrinkle: Ollama's installer unpacks a `.tar.zst` archive, and Colab's
image doesn't include `zstd`. We install that first, or the install fails.

In [ ]:
# Ollama's installer needs zstd to unpack its archive, and Colab
# doesn't ship it. Install it first.
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

Now install Ollama itself.

In [ ]:
# Takes about a minute. The output is noisy - that's normal.
!curl -fsSL https://ollama.com/install.sh | sh

### A2. Start the Ollama server

Ollama runs as a background *server*; our Python code talks to it over HTTP.

We start it with `subprocess.Popen` rather than `!ollama serve`, because a `!`
command would block this cell forever. `Popen` launches it in the background
and lets the notebook carry on.

In [ ]:
import subprocess
import time
import urllib.request

# Launch the server in the background, silencing its logs so they don't
# flood the notebook.
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)


def ollama_is_running():
    """Return True once the server is accepting connections."""
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False


# Poll until it's actually up, rather than guessing with a fixed sleep.
for attempt in range(30):
    if ollama_is_running():
        print("Ollama server is up.")
        break
    time.sleep(1)
else:
    print("Server didn't start in 30s. Try re-running this cell.")
    print("If it keeps failing, check that the A1 cells finished without errors.")

### A3. Download the model

`llama3.2:3b` is a 3-billion-parameter model — about **2 GB**, so this is the
slowest step. It's small enough to be fast, but far smaller than the models
behind commercial chat products. Part C is about exactly what that costs you.

In [ ]:
!ollama pull llama3.2:3b

### A4. Check everything is working

This confirms the server is reachable *and* the model is present — the two
things everything below depends on. If it reports a problem, re-run the cell
it points you to.

In [ ]:
import json
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as r:
        installed = [m["name"] for m in json.loads(r.read()).get("models", [])]
except Exception:
    installed = None

if installed is None:
    print("[X] Can't reach the Ollama server. Re-run cell A2.")
elif not any(m.startswith("llama3.2:3b") for m in installed):
    print(f"[X] Server is up, but llama3.2:3b isn't installed. Found: {installed}")
    print("    Re-run cell A3.")
else:
    print("[OK] Server is running and llama3.2:3b is ready.")
    print(f"     Installed models: {', '.join(installed)}")
    print("\nSetup complete - continue to Part B.")

# Part B — The code

Everything below talks to Ollama over its plain HTTP API using only the Python
standard library. No SDK, no LangChain, no framework.

That's deliberate: every request is written out in full, so you can read this
top to bottom and see exactly what goes to the model and what comes back.

### The one function that talks to the model

Read this cell carefully — it's the whole integration, and every part of this
notebook uses it. It takes three optional knobs, and each Part turns a
different one:

| argument | used in | controls |
| --- | --- | --- |
| `options` | Part B | *how* the text is generated (randomness, length) |
| `system` | Part C | *how the model should behave* while answering |
| `response_format` | Part D | *the shape* of the reply (plain text vs. JSON) |

In [ ]:
import json
import urllib.request

# Where Ollama is listening. Because the server runs on this same Colab
# machine, it's the normal local address.
OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:3b"


def call_ollama(prompt, model=MODEL, system=None, options=None, response_format=None):
    """Send a prompt to Ollama and return the model's reply as a string.

    Arguments:
        prompt          The question or task (the "user prompt").
        model           Which model to use.
        system          Optional instructions about HOW to answer - tone,
                        role, format, constraints. Used in Part C.
        options         A dict of generation parameters, e.g.
                        {"temperature": 0.8, "seed": 42}. Used in Part B.
        response_format Constrains the OUTPUT FORMAT. Either "json", or a
                        JSON Schema dict. Used in Part D.
    """
    # Build the JSON body Ollama expects. stream=False means we get one
    # complete response instead of a trickle of partial tokens.
    payload = {"model": model, "prompt": prompt, "stream": False}

    # Only include the optional fields if they were actually provided, so we
    # send the smallest, clearest request possible.
    if system:
        payload["system"] = system
    if options:
        payload["options"] = options
    if response_format is not None:
        payload["format"] = response_format

    # Encode as JSON bytes and POST to the /api/generate endpoint.
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=data,
        headers={"Content-Type": "application/json"},
    )

    # The reply is a JSON object; the text we want is under "response".
    with urllib.request.urlopen(request, timeout=120) as response:
        body = json.loads(response.read().decode("utf-8"))
    return body["response"]


print("call_ollama() is ready.")

## B.1. The smallest possible call

One prompt in, one string out. No options at all, so the model uses its
defaults.

**Try it:** change the prompt and re-run.

In [ ]:
prompt = "In one sentence, what is a large language model?"

reply = call_ollama(prompt)

print(reply.strip())

## B.2. `temperature`: the randomness dial

Temperature controls how much the model is allowed to gamble when picking each
next word.

| Setting | Behavior |
| --- | --- |
| Low (0.0–0.3) | Nearly deterministic. Picks the most likely word almost every time. Good for facts and code. |
| High (0.8–1.5) | Adventurous. Willing to pick less likely words — more creative, more likely to go off the rails. |

The cell below sends the **same prompt twice** at each setting. At low
temperature the two runs look nearly identical; at high temperature they drift
apart. That divergence *is* the randomness.

In [ ]:
prompt = "Write a one-sentence bedtime story about a sleepy robot."

for temperature in (0.0, 1.2):
    print(f"--- temperature = {temperature} ---")

    # Building this dict is the entire point of the step.
    options = {"temperature": temperature}

    first_run = call_ollama(prompt, options=options)
    second_run = call_ollama(prompt, options=options)

    print(f"  Run 1: {first_run.strip()}")
    print(f"  Run 2: {second_run.strip()}")

    if first_run.strip() == second_run.strip():
        print("  -> The two runs are IDENTICAL.\n")
    else:
        print("  -> The two runs DIFFER.\n")

## B.3. `top_p` and `top_k`: narrowing the candidate pool

Temperature decides *how much* to gamble. These two decide *which words are
even on the table* before that gamble happens. At each step the model has a
ranked list of possible next words, and these trim it:

- **`top_k`** — keep only the *k* most likely words. `top_k=1` forces the
  single best candidate every time, which makes output safe and repetitive.
- **`top_p`** — keep the smallest set of top words whose probabilities sum to
  *p*. `top_p=0.9` means "the most likely words covering 90% of the
  probability." Also called *nucleus sampling*.

Lower values are safer for both.

In [ ]:
prompt = "Suggest a creative name for a campus coffee shop."

# Two deliberately extreme settings, so the contrast is obvious.
settings = [
    ("very narrow", {"top_k": 1, "top_p": 0.1, "temperature": 1.0}),
    ("wide open",   {"top_k": 100, "top_p": 1.0, "temperature": 1.0}),
]

for label, options in settings:
    print(f"--- {label}: {options} ---")
    reply = call_ollama(prompt, options=options)
    print(f"  {reply.strip()}\n")

## B.4. `seed`: making randomness repeatable

Temperature makes output random — but sometimes you want *repeatable*
randomness, so a demo or a test gives the same result every time. The **seed**
fixes the starting point of the random number generator:

> same seed + same parameters + same prompt → the same output

We use a **high temperature** here on purpose. Without a seed, these two
replies would almost certainly differ — so if they come back identical, that's
the seed doing the work.

In [ ]:
prompt = "Invent a name and one-line backstory for a friendly dragon."

# High temperature, so the seed's effect is unmistakable.
options = {"temperature": 1.0, "seed": 42}

first_run = call_ollama(prompt, options=options)
second_run = call_ollama(prompt, options=options)

print(f"--- same seed, twice: {options} ---")
print(f"  Run 1: {first_run.strip()}")
print(f"  Run 2: {second_run.strip()}")

if first_run.strip() == second_run.strip():
    print("  -> IDENTICAL. The seed made the randomness repeatable.\n")
else:
    print("  -> Not identical. Unusual, but some runtime settings can")
    print("     still introduce small differences.\n")

# Now change ONLY the seed. Different answer - but equally repeatable.
different = {"temperature": 1.0, "seed": 12345}
third_run = call_ollama(prompt, options=different)
print(f"--- different seed: {different} ---")
print(f"  Run 3: {third_run.strip()}")
print("  -> A different answer, because the seed changed. Re-run this cell")
print("     and Run 3 will reproduce exactly.")

## B.5. `num_predict` and `stop`: controlling length

Two dials that decide *when the model stops talking*:

- **`num_predict`** — a hard cap on how many tokens (roughly, word pieces) the
  model may generate. It's also the main knob for keeping a slow model
  responsive.
- **`stop`** — a list of strings that, if generated, cut output off
  immediately. Useful for structured output: stop at `"\n"` for a single
  line, or at a marker like `"END"`.

**Important:** `num_predict` does *not* ask the model to be brief. It just
stops generation when the cap is hit — which is why the output truncates
mid-sentence.

In [ ]:
prompt = "List three tips for studying effectively."

# A very small cap, to make the truncation obvious.
print("--- num_predict = 16 (very short cap) ---")
print(call_ollama(prompt, options={"num_predict": 16}).strip())
print("  -> Likely cuts off mid-thought. It wasn't asked to be brief;")
print("     it was simply stopped.\n")

# A roomier cap, for comparison.
print("--- num_predict = 200 (room to finish) ---")
print(call_ollama(prompt, options={"num_predict": 200}).strip(), "\n")

# A stop sequence: halt at the first line break, forcing a single line.
print("--- stop = ['\\n'] (stop at the first line break) ---")
print(call_ollama(prompt, options={"num_predict": 200, "stop": ["\n"]}).strip())
print("  -> Generation halted the moment a newline appeared.")

## B.6. Your turn

Fill in the three values below so the answer is:

- **highly creative / random** — think `temperature`
- **repeatable across runs** — think `seed`
- **capped to a short length** — think `num_predict`

Replace each `None` with a number, then run the cell. It calls the model
*twice*, so you can confirm your seed makes the answer repeat.

The solution is in the next cell — try it yourself first.

In [ ]:
# ---------------------------------------------------------------------
# TODO: replace each None with a number.
#   temperature - high for creativity          (try ~1.2)
#   seed        - any integer, so runs repeat  (try 7)
#   num_predict - small, to keep it short      (try 40)
# ---------------------------------------------------------------------
my_options = {
    "temperature": None,   # TODO
    "seed": None,          # TODO
    "num_predict": None,   # TODO
}

prompt = "Give me a fun team name for a robotics club."

# Drop any blanks still set to None, so the cell runs while you're partway
# through the exercise.
filled_in = {key: value for key, value in my_options.items() if value is not None}

if not filled_in:
    print("The TODOs aren't filled in yet - calling with pure defaults.\n")

print(f"Options being used: {filled_in}\n")

first_run = call_ollama(prompt, options=filled_in)
second_run = call_ollama(prompt, options=filled_in)
print(f"  Run 1: {first_run.strip()}")
print(f"  Run 2: {second_run.strip()}")

# Check your work.
goals_met = []
if filled_in.get("temperature", 0) and filled_in["temperature"] >= 0.8:
    goals_met.append("creative (high temperature)")
if "seed" in filled_in and first_run.strip() == second_run.strip():
    goals_met.append("repeatable (seed fixed, both runs matched)")
if filled_in.get("num_predict") and filled_in["num_predict"] <= 60:
    goals_met.append("short (low num_predict)")

if len(goals_met) == 3:
    print("\n[OK] All three goals met: " + "; ".join(goals_met))
else:
    print(f"\n[TODO] Goals met so far: {goals_met or 'none'}")

### (Solution)

```python
my_options = {
    "temperature": 1.2,   # high -> creative / random
    "seed": 7,            # fixed -> repeatable across runs
    "num_predict": 40,    # small -> capped length
}
```

With these, both runs return the **same** short, creative answer. High
temperature supplies the creativity, the seed makes that particular creative
result repeatable, and `num_predict` keeps it brief. Change the seed and you
get a different — but again repeatable — answer.

### Part B recap

| Dial | What it controls |
| --- | --- |
| `temperature` | how random / creative the output is |
| `top_p`, `top_k` | which candidate words are even considered |
| `seed` | makes random output repeatable |
| `num_predict` | hard cap on output length |
| `stop` | strings that cut generation off early |

Every one of these is just a key in the `options` dictionary passed to
`call_ollama()`. That's the whole idea.

# Part C — Prompts, limits, and tools

Part B controlled *how* the model generates text. This Part is about *what it
can and can't do*, in three moves:

1. **System prompts** — the cheapest lever on model behavior.
2. **Where small models break down** — five failure modes, on purpose.
3. **Tool use** — fixing one of those failures with real code.

## C.1. System prompt vs. user prompt

Two kinds of text go into a request, and they do different jobs:

- **user prompt** — the actual question or task.
- **system prompt** — instructions about *how* to behave while answering:
  tone, role, format, constraints.

The cell below asks **one fixed question** under several different system
prompts. Watch the tone, structure, and even the *content* shift while the
question never changes.

This is the cheapest, fastest lever you have on model behavior — no
retraining, no RAG, just a different instruction string.

In [ ]:
# Each of these is an instruction about HOW to answer, not WHAT to answer.
SYSTEM_PROMPTS = {
    "(none - model default)": None,
    "Helpful assistant": (
        "You are a helpful, friendly assistant. Answer clearly and concisely."
    ),
    "Terse domain expert": (
        "You are a terse subject-matter expert. Answer in bullet points, no "
        "preamble, no pleasantries, no hedging."
    ),
    "Explain like I'm 5": (
        "You are explaining things to a curious 5-year-old. Use simple words, "
        "short sentences, and a concrete everyday example."
    ),
    "Skeptical scientist": (
        "You are a skeptical scientist. For every claim, note the evidence "
        "quality and what could make it wrong. Avoid overconfidence."
    ),
    "Pirate": (
        "You are a pirate captain. Answer entirely in pirate dialect, but keep "
        "the actual information accurate."
    ),
}

# The user prompt is held completely fixed. Only `system` varies.
question = "Should I refactor this messy code now, or ship the feature first?"
print(f"User prompt (identical every time): {question}\n")

for name, system_prompt in SYSTEM_PROMPTS.items():
    print(f"--- system prompt: {name} ---")
    reply = call_ollama(question, system=system_prompt)
    print(f"  {reply.strip()}\n")

## C.2. Where local models break down

`llama3.2:3b` has **3 billion parameters** — small enough to run on a free
Colab VM, but far smaller than the 100B+ models behind commercial chat
products. That gap shows up as specific, learnable failure modes.

The cell below runs five of them on purpose, each with an explanation of *why*
it breaks.

Read the answers **carefully**. These failures are usually wrong in a fluent,
confident way rather than an obviously broken way — which is exactly what
makes them worth seeing.

In [ ]:
BREAKDOWN_EXAMPLES = [
    {
        "name": "Letter counting (tokenization blindness)",
        "prompt": "How many times does the letter 'r' appear in the word 'strawberry'?",
        "why": (
            "LLMs don't see individual letters. Text is split into sub-word "
            "TOKENS before the model sees it, so 'strawberry' might be 2-3 "
            "opaque chunks rather than 10 characters."
        ),
    },
    {
        "name": "Multi-digit arithmetic",
        "prompt": "What is 84,637 multiplied by 92,481? Give the exact number.",
        "why": (
            "Small models pattern-match arithmetic from training examples "
            "rather than running an algorithm - there's no built-in "
            "calculator. C.3 fixes exactly this."
        ),
    },
    {
        "name": "Knowledge cutoff (recent events)",
        "prompt": "Who won the most recent Super Bowl, and what was the final score?",
        "why": (
            "The model only knows what was in its training data, which has a "
            "cutoff date. Ask about anything later and it may confidently "
            "guess from old patterns."
        ),
    },
    {
        "name": "Multi-constraint instructions",
        "prompt": (
            "Write exactly three sentences about coffee. The first sentence "
            "must start with the letter B. The second sentence must contain "
            "exactly seven words. Do not use the word 'bean' anywhere."
        ),
        "why": (
            "Small models tend to drop one constraint when asked to satisfy "
            "several at once. Watch WHICH one it fails - and how confidently "
            "it claims to have followed them all."
        ),
    },
    {
        "name": "Confident fabrication (hallucination)",
        "prompt": "Summarize the plot of the novel 'The Glass Meridian' by Aldous Whitfield.",
        "why": (
            "This book and author don't exist. A well-behaved model says so. "
            "A model prone to hallucination invents a plausible summary - a "
            "reminder that fluent text is not the same as true text."
        ),
    },
]

for example in BREAKDOWN_EXAMPLES:
    print(f"--- {example['name']} ---")
    print(f"  Prompt: {example['prompt']}")
    reply = call_ollama(example["prompt"])
    print(f"  Reply:  {reply.strip()}")
    print(f"  WHY IT BREAKS: {example['why']}\n")

## C.3. Tool use — giving the model a calculator

C.2 showed the model guessing at multi-digit arithmetic and getting it wrong
with total confidence. **Tool use** fixes that whole class of problem.

Instead of asking the model to *compute* an answer, we teach it to *ask for* a
calculation, then run real Python to get the exact number:

1. **Ask** — with a system prompt saying it may reply `CALC: <expression>`
   instead of guessing.
2. **Check and calculate** — if the reply starts with `CALC:`, pull out the
   expression and evaluate it in Python.
3. **Ask again** — hand the exact result back so the final answer uses real
   math.

This is the same idea as "function calling" in bigger frameworks, written out
by hand so every step is visible.

In [ ]:
import re

TOOL_SYSTEM_PROMPT = (
    "You have access to a calculator tool for arithmetic. If answering the "
    "question requires a calculation, respond with ONLY a single line in the "
    "exact form 'CALC: <expression>' using plain + - * / ** and parentheses "
    "(e.g. 'CALC: 84637 * 92481') - no commas in numbers, no explanation, no "
    "extra text. If the question does NOT require a calculation, just answer "
    "it directly and normally."
)


def safe_calculate(expression):
    """Evaluate a plain arithmetic expression like '84637 * 92481'.

    We can't hand model-generated text to Python's real eval() - that would
    let it run arbitrary code. So we check the string contains ONLY digits,
    decimal points, whitespace and + - * / ** ( ), and only then eval() with
    builtins disabled. Between the whitelist and the disabled builtins,
    there's nothing left to do but arithmetic.

    Note on commas: the system prompt asks for numbers without them, but the
    model doesn't always comply - especially when the question itself used
    them ("84,637"). We strip commas rather than fail; between digits a comma
    is unambiguous.
    """
    expression = expression.replace(",", "")

    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        raise ValueError(f"Expression contains disallowed characters: {expression!r}")

    return eval(expression, {"__builtins__": {}}, {})


def ask_with_calculator(question):
    """Run the full ask -> check -> calculate -> ask-again loop."""
    # Step 1: ask, offering the tool.
    first_reply = call_ollama(question, system=TOOL_SYSTEM_PROMPT)
    print(f"  [1] Model's first reply: {first_reply.strip()}")

    # Step 2: did it request a calculation? If not, its reply is final.
    if not first_reply.strip().upper().startswith("CALC:"):
        print("  [2] No calculation requested - first reply is the final answer.")
        return first_reply

    expression = first_reply.strip().split(":", 1)[1].strip()
    try:
        exact_result = safe_calculate(expression)
    except (ValueError, SyntaxError, ZeroDivisionError) as error:
        print(f"  [2] Tool call FAILED on {expression!r}: {error}")
        return f"(Could not safely evaluate {expression!r})"

    print(f"  [2] Tool called: {expression}  ->  exact result: {exact_result}")

    # Step 3: hand the real number back for a final answer.
    follow_up = (
        f"Question: {question}\n"
        f"You requested the calculation `{expression}`, and the exact result "
        f"is {exact_result}. Give the final answer using this exact result - "
        f"do not recompute it yourself."
    )
    final_answer = call_ollama(follow_up)
    print("  [3] Sent the exact result back for a final answer.")
    return final_answer


print("Calculator tool ready.")

Now compare the same question **with and without** the tool.

In [ ]:
question = "What is 84,637 multiplied by 92,481?"
correct_answer = 84637 * 92481

print(f"Question: {question}")
print(f"(The true answer, computed by Python: {correct_answer})\n")

# --- Without the tool: the C.2 failure, reproduced ---
print("--- WITHOUT the calculator tool ---")
guess = call_ollama(question)
print(f"  {guess.strip()}")
if str(correct_answer) in guess:
    print("  -> It got it right this time. Re-run; it won't last.\n")
else:
    print(f"  -> WRONG. The correct answer is {correct_answer}.\n")

# --- With the tool ---
print("--- WITH the calculator tool ---")
answer = ask_with_calculator(question)
print(f"\n  Final answer: {answer.strip()}")
if str(correct_answer) in answer:
    print("  -> CORRECT. Real Python did the math, not the model.\n")
else:
    print("  -> The exact number didn't survive into the wording, but the")
    print("     tool result itself was exact.\n")

# --- A question needing no arithmetic: the model should skip the tool ---
print("--- A question that needs NO calculation ---")
no_math = ask_with_calculator("What is the capital of France?")
print(f"\n  Final answer: {no_math.strip()}")

### Part C recap

1. **System prompt** — instructions for *how* to answer. Free to change, no
   retraining, and it shifts tone, format, even willingness to hedge.
2. **Capability ceiling** — no amount of prompt engineering fixes tokenization
   blindness or a knowledge cutoff. Some failures need a bigger model; some
   need external tools.
3. **Tool use** — don't ask the model to compute, ask it to *delegate* to code
   that actually computes. The same pattern extends to a search tool, a
   database tool, or a document retrieval tool — which is what RAG is,
   structurally.

# Part D — Structured output

So far we've controlled *what* the model says and *how randomly* it says it.
This Part controls the **format** of the answer: how do you get back clean,
machine-readable data — JSON with exactly the fields you need — instead of a
paragraph you have to parse by hand?

Four steps, each fixing the previous one's weakness:

| Step | Approach | What you get |
| --- | --- | --- |
| D.1 | Naive prompt | maybe-valid text |
| D.2 | `format="json"` | valid JSON, any shape |
| D.3 | JSON Schema | valid JSON, *your* shape |
| D.4 | Pydantic | typed, validated object |

This is where `call_ollama`'s third knob — `response_format` — comes in.

## D.1. Naive prompt — just ask for JSON

The obvious first attempt: write "respond in JSON" in the prompt and hope.

The cell below does that with **no `format` field**, then tries to
`json.loads()` the reply. Run it a few times. Common failures: the model wraps
the JSON in prose ("Here is the JSON you requested:"), fences it in markdown,
or adds a trailing comma — any of which breaks a strict parser.

In [ ]:
prompt = (
    "Give me three programming languages, each with a one-word "
    "description. Respond in JSON."
)

raw_reply = call_ollama(prompt)  # no response_format

print("Raw reply from the model:")
print("-" * 60)
print(raw_reply)
print("-" * 60)

try:
    parsed = json.loads(raw_reply)
    print("\n[OK] json.loads() succeeded this time:")
    print(json.dumps(parsed, indent=2))
    print("\n...but that's luck, not a guarantee. Run it again a few times.")
except json.JSONDecodeError as error:
    print(f"\n[FAIL] json.loads() could not parse the reply: {error}")
    print("The model returned text, not clean JSON. D.2 fixes this.")

## D.2. `format="json"` — guarantee *valid* JSON

Same prompt, but now we pass `response_format="json"`. Ollama constrains the
model's decoding so the output **must** be syntactically valid JSON — no
prose, no fences, no trailing commas. `json.loads()` will succeed.

The catch: valid JSON is not the *same* JSON every time. The model still picks
its own keys, so one run might give `{"languages": [...]}` and another a bare
list. Valid, but unpredictable — which is what D.3 fixes.

In [ ]:
raw_reply = call_ollama(prompt, response_format="json")

parsed = json.loads(raw_reply)  # safe: format="json" guarantees this parses

print("Valid JSON, guaranteed to parse:")
print(json.dumps(parsed, indent=2))
print("\nNotice which top-level keys the model chose. Re-run and they may")
print("well change - valid JSON, but not a shape you can rely on.")

## D.3. JSON Schema — lock down the *shape*

Now we hand Ollama an actual **JSON Schema** as the format. The output is
constrained to match it: the exact fields you name, with the types you
declare, and every `required` key present.

This is the real workhorse of structured output — pulling reliable,
predictable records out of free text.

> **Version note:** schema-constrained output needs a reasonably recent
> Ollama. If the cell reports an error instead of a result, the installed
> version is too old — `format="json"` from D.2 still works everywhere.

In [ ]:
import urllib.error

# A JSON Schema is just a dictionary describing the structure you want.
PERSON_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "city": {"type": "string"},
        "occupation": {"type": "string"},
    },
    # Fields listed here MUST be present in the output.
    "required": ["name", "age", "city", "occupation"],
}

text = "Maria is a 34-year-old software engineer who lives in Austin, Texas."
print(f"Text to extract from:\n  {text}\n")

try:
    raw_reply = call_ollama(
        f"Extract the person's details from this text:\n\n{text}",
        response_format=PERSON_SCHEMA,
    )
    parsed = json.loads(raw_reply)
    print("Extracted, conforming to the schema:")
    print(json.dumps(parsed, indent=2))

    missing = [f for f in PERSON_SCHEMA["required"] if f not in parsed]
    if missing:
        print(f"\n[WARN] Missing required field(s): {missing}")
    else:
        print("\n[OK] Every required field is present - same keys, every run.")
        print(f"     Because the shape is guaranteed, parsed['age'] = {parsed['age']}")
except urllib.error.HTTPError as error:
    print(f"[FAIL] Ollama rejected the schema (HTTP {error.code}).")
    print("This Ollama is probably too old for schema-constrained output.")
    print('D.2 (format="json") still works.')

## D.4. Pydantic — a typed, *validated* Python object

JSON Schema gets you the right shape on the wire. **Pydantic** takes the last
step: define the shape once as a typed Python class, let it *generate* the
schema, then *validate* the reply back into a real object.

Two payoffs over raw schema:

- **One source of truth** — the class produces the schema, so they can't drift
  apart.
- **Validation** — if anything comes back malformed, it raises instead of
  silently handing you bad data.

In [ ]:
!pip install -q pydantic

In [ ]:
from pydantic import BaseModel, ValidationError


# The shape we want, written once as a normal typed Python class.
# (Deliberately no docstring: pydantic copies docstrings into the generated
# schema as a "description" field, which is just noise when we print it.)
class Product(BaseModel):
    name: str
    price_usd: float
    in_stock: bool
    tags: list[str]


text = (
    "The UltraGrip water bottle costs $24.99, is currently in stock, and "
    "is great for hiking, cycling, and gym use."
)

# Generate the JSON Schema straight from the class definition.
schema = Product.model_json_schema()
print("Schema auto-generated from the Product class:")
print(json.dumps(schema, indent=2))

try:
    raw_reply = call_ollama(
        f"Extract the product details from this text:\n\n{text}",
        response_format=schema,
    )
    # Parse AND validate into a real Product object in one step.
    product = Product.model_validate_json(raw_reply)

    print("\nValidated Product object - now real Python types, not text:")
    print(f"  product.name      = {product.name!r}  ({type(product.name).__name__})")
    print(f"  product.price_usd = {product.price_usd!r}  ({type(product.price_usd).__name__})")
    print(f"  product.in_stock  = {product.in_stock!r}  ({type(product.in_stock).__name__})")
    print(f"  product.tags      = {product.tags!r}  ({type(product.tags).__name__})")

    # Because these are real types, we can use them directly.
    print(f"\n  Arithmetic works: two bottles cost ${product.price_usd * 2:.2f}")
    print(f"  Iteration works:  {len(product.tags)} tags -> {', '.join(product.tags)}")
except ValidationError as error:
    print(f"\n[FAIL] Validation caught malformed data: {error}")
    print("That's the point - bad data is caught here, not downstream.")
except urllib.error.HTTPError as error:
    print(f"\n[FAIL] Ollama rejected the schema (HTTP {error.code}) - likely too old.")

## D.5. Your turn — write a schema

Below is a half-finished JSON Schema for extracting a **recipe**. Fill in the
`# TODO` fields so the schema requires:

- `title` — a string (done for you as an example)
- `servings` — an **integer**
- `ingredients` — an **array of strings**

Hints:

```python
an integer        ->  {"type": "integer"}
a list of strings ->  {"type": "array", "items": {"type": "string"}}
```

Remember to add the new field names to the `"required"` list too. The solution
is in the next cell.

In [ ]:
# ---------------------------------------------------------------------
# TODO: replace the two None values, and extend "required".
# ---------------------------------------------------------------------
my_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},   # example - leave this one alone
        "servings": None,              # TODO: an integer
        "ingredients": None,           # TODO: a list of strings
    },
    "required": ["title"],             # TODO: add "servings" and "ingredients"
}

recipe_text = (
    "Classic guacamole serves 4. You'll need avocados, lime juice, salt, "
    "onion, and cilantro."
)

# Drop fields still set to None so the schema stays valid while you work.
# We filter "required" the same way - naming a required field that has no
# definition is an invalid schema, and the error would look like a version
# problem rather than a half-finished exercise.
filled_in = {k: v for k, v in my_schema["properties"].items() if v is not None}
working_schema = {
    "type": "object",
    "properties": filled_in,
    "required": [f for f in my_schema["required"] if f in filled_in],
}

if len(filled_in) < 3:
    print(f"TODOs not filled in yet - only extracting: {', '.join(filled_in)}\n")

try:
    raw_reply = call_ollama(
        f"Extract the recipe details from this text:\n\n{recipe_text}",
        response_format=working_schema,
    )
    parsed = json.loads(raw_reply)
    print("Extracted with your schema:")
    print(json.dumps(parsed, indent=2))

    if all(f in parsed for f in ("title", "servings", "ingredients")):
        print("\n[OK] All three fields came back - schema complete. Nice work.")
    else:
        print("\n[TODO] Still missing 'servings' or 'ingredients'.")
except urllib.error.HTTPError as error:
    print(f"[FAIL] Ollama rejected the schema (HTTP {error.code}) - likely too old.")

### (Solution)

```python
my_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "servings": {"type": "integer"},
        "ingredients": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["title", "servings", "ingredients"],
}
```

An `array` needs an `items` entry describing what each element looks like.
Adding all three names to `required` forces the model to return every one, so
`servings` comes back as a real integer and `ingredients` as a real list.

---

# Where this leaves you

| Part | Lever | What it controls |
| --- | --- | --- |
| **B** | `options` | how the text is generated — randomness, length, repeatability |
| **C** | `system` | how the model behaves — and where it simply can't help |
| **C** | tools | delegating what the model is bad at to real code |
| **D** | `response_format` | the shape of what comes back |

All four run through the same small `call_ollama()` function you read at the
top of Part B. There's no framework here — just one HTTP request with
different fields set.

**Next: RAG.** Tool use in C.3 gave the model a calculator. Retrieval-Augmented
Generation applies the same pattern with a *document retrieval* tool: before
answering, the model gets relevant passages from a corpus it was never trained
on. That's a separate notebook.

## If you want to go further

The full workshop repository has all of this as marimo notebooks (interactive
sliders and buttons) and as plain Python scripts, plus a guide to installing
Ollama on your own machine — where, unlike Colab, the "nothing leaves this
machine" promise actually holds.